# EDA för swedavia.duckdb

In [ ]:
# pip install duckdb pandas
import duckdb
import pandas as pd

con = duckdb.connect("data/swedavia.duckdb", read_only=True)

# Hämta en tabell till pandas
df = con.sql("SELECT * FROM raw.flights").df()

### Egen funktion som returnerar vad som kollas och hur mycket som är "fel"

In [6]:
def check(namn, antal):
    print(f"{'OK ' if antal == 0 else 'OBS'}  {namn}: {antal}")

### Dela upp tabellen så den har arrivals och departures

In [5]:
arr = df[df.direction == "arrivals"]
dep = df[df.direction == "departures"]

# EDA datakvalitet

In [7]:
# Dubbletter och saknade nyckelvärden
check("helt identiska rader", df.drop(columns=["_dlt_id", "_dlt_load_id"]).duplicated().sum())
for c in ["flight_id", "airport_iata", "direction", "query_date"]:
    check(f"saknar {c}", df[c].isna().sum())

OK   helt identiska rader: 0
OK   saknar flight_id: 0
OK   saknar airport_iata: 0
OK   saknar direction: 0
OK   saknar query_date: 0


In [8]:
# Flygplatskoder ska vara exakt 3 stora bokstäver
for c in ["flight_leg_identifier__departure_airport_iata",
          "flight_leg_identifier__arrival_airport_iata"]:
    check(f"{c} ej 3 bokstäver", (~df[c].str.fullmatch(r"[A-Z]{3}", na=False)).sum())

OK   flight_leg_identifier__departure_airport_iata ej 3 bokstäver: 0
OK   flight_leg_identifier__arrival_airport_iata ej 3 bokstäver: 0


In [9]:
# Tider: skillnad mellan faktisk och schemalagd tid, i minuter
diff_arr = (arr.arrival_time__actual_utc - arr.arrival_time__scheduled_utc).dt.total_seconds() / 60
diff_dep = (dep.departure_time__actual_utc - dep.departure_time__scheduled_utc).dt.total_seconds() / 60
check("ankomster som skiljer mer än 6 h från schema", (diff_arr.abs() > 360).sum())
check("avgångar som skiljer mer än 6 h från schema", (diff_dep.abs() > 360).sum())
check("gate stänger före den öppnar",
      (dep.location_and_status__gate_open_utc > dep.location_and_status__gate_close_utc).sum())

OBS  ankomster som skiljer mer än 6 h från schema: 7
OBS  avgångar som skiljer mer än 6 h från schema: 7
OBS  gate stänger före den öppnar: 4


In [10]:
check("LAN utan faktisk ankomsttid",
      (arr.location_and_status__flight_leg_status.eq("LAN") & arr.arrival_time__actual_utc.isna()).sum())
check("ACT utan faktisk avgångstid",
      (dep.location_and_status__flight_leg_status.eq("ACT") & dep.departure_time__actual_utc.isna()).sum())

OK   LAN utan faktisk ankomsttid: 0
OK   ACT utan faktisk avgångstid: 0


In [11]:
check("ankomst som inte landar på airport_iata",
      (arr.flight_leg_identifier__arrival_airport_iata != arr.airport_iata).sum())
check("avgång som inte startar från airport_iata",
      (dep.flight_leg_identifier__departure_airport_iata != dep.airport_iata).sum())

OK   ankomst som inte landar på airport_iata: 0
OK   avgång som inte startar från airport_iata: 0


In [12]:
# Gate-värden: titta på alla unika
sorted(df.location_and_status__gate.dropna().unique())

['01',
 '02',
 '03',
 '04',
 '05',
 '1',
 '10',
 '12',
 '13',
 '13C',
 '14',
 '14C',
 '15',
 '16',
 '17',
 '18D',
 '18E',
 '18F',
 '18G',
 '19',
 '2',
 '20A',
 '20B',
 '21A',
 '21B',
 '21X',
 '22',
 '22X',
 '23',
 '24B',
 '24D',
 '3',
 '30',
 '33',
 '4',
 '5',
 '61A',
 '61B',
 '61C',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '7',
 '86',
 '87',
 '9A',
 'A1',
 'A108',
 'A112',
 'A113',
 'A115',
 'A119',
 'A3',
 'A4',
 'BUSS',
 'C31',
 'C33',
 'C35',
 'C37',
 'C39',
 'C41',
 'C43',
 'D11',
 'D12',
 'D13',
 'D14',
 'D14B',
 'D15',
 'D16',
 'D18',
 'D20',
 'E1',
 'E10',
 'E3',
 'E3B',
 'E4',
 'E5',
 'E5B',
 'E6',
 'E6B',
 'E7',
 'E8',
 'E9',
 'F26',
 'F28',
 'F29',
 'F3',
 'F30',
 'F31',
 'F32',
 'F33',
 'F34',
 'F35',
 'F36',
 'F37',
 'F38',
 'F39',
 'F5',
 'F58',
 'F59',
 'F60',
 'F62',
 'F63',
 'F65',
 'F66',
 'F67',
 'F69',
 'I',
 'J',
 'K',
 'M',
 'PASS',
 'VIP']